## 04 — Vector Stores
- Build FAISS + Chroma indexes and visualize the embedding space.

In [ ]:
import sys
import os
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap

from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.ingestion import load_documents
from rag_pipeline.splitting import split_documents
from rag_pipeline.vectorstores import build_vectorstore

REPO = "/content/a-survey-of-retrieval-augmented-generation-RAG"
CSV  = f"{REPO}/data/arxiv_data.csv"

In [ ]:
# Load with the correct column 
docs = load_documents([{
    "type": "csv",
    "path": CSV,
    "content_columns": ["summaries"],       # ← fixed
}])[:500]

print(f"Loaded {len(docs)} docs, first has {len(docs[0].page_content)} chars")
assert len(docs[0].page_content) > 0, "Empty docs — wrong content_columns?"

In [ ]:
# Chunk
chunks = split_documents(docs, {
    "type": "recursive", "chunk_size": 1000, "chunk_overlap": 200,
})
print(f"Split into {len(chunks)} chunks")
assert len(chunks) > 0, "No chunks — check content_columns again."

In [ ]:
# Embeddings
emb = build_embeddings({"provider": "huggingface", "model": "BAAI/bge-small-en-v1.5"})

In [ ]:
# FAISS
faiss_store = build_vectorstore(chunks, emb, {
    "type": "faiss",
    "persist_dir": f"{REPO}/indexes/faiss_index",  
})
print("FAISS total:", faiss_store.index.ntotal)

In [ ]:
# Chroma
chroma_store = build_vectorstore(chunks, emb, {
    "type": "chroma",
    "persist_dir": f"{REPO}/indexes/chroma_arxiv",
    "collection_name": "arxiv",
    "metric": "cosine",
})
print("Chroma count:", chroma_store._collection.count())

In [ ]:
# PCA + UMAP visualization 
texts = [c.page_content for c in chunks]
matrix = np.array(emb.embed_documents(texts))
print("Embedding matrix:", matrix.shape)

pca = PCA(n_components=2, random_state=42).fit_transform(matrix)
umap_2d = umap.UMAP(
    n_neighbors=15, n_components=2, metric="cosine",
    min_dist=0.1, random_state=42,
).fit_transform(matrix)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.scatter(pca[:, 0], pca[:, 1], s=15, alpha=0.7); ax1.set_title("PCA")
ax2.scatter(umap_2d[:, 0], umap_2d[:, 1], s=15, alpha=0.7);